# Notebook 5 — Self-Quiz

**20 questions covering everything from N1 to N4.** Mix of multiple-choice and predict-the-state. Each question is auto-checked. Final score at the end.

## How this works

Each question has two cells:
1. A **predict cell** where you assign your answer to a variable.
2. A **check cell** that tells you right/wrong with a brief explanation.

**Don't run the check cell until you've made a real attempt.** A blank prediction (leaving `...`) counts as wrong.

## Sections

- **A** — Syntax & Grammar (Q1–4)
- **B** — States and updates (Q5)
- **C** — Evaluators `A` and `B` (Q6–7)
- **D** — Small-step rules (Q8–11)
- **E** — Tracing programs (Q12–14, Q19–20)
- **F** — Reasoning about the semantics (Q15–18)

Run all cells **in order** — the score tracker is a stateful global.

In [ ]:
import sys
from pathlib import Path

# Make this work whether jupyter was started from notebooks/ or from the repo root.
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py — start jupyter from the notebooks/ folder")

from while_lang import parse, run, trace, A, B, step, step_iter, Config, StepBudgetExceeded

# Score tracker.
_RESULTS: dict[int, bool] = {}

def check(qnum: int, predicted, actual, explain: str = '') -> None:
    """Compare predicted vs actual; record + report. Ellipsis = unanswered."""
    if predicted is Ellipsis:
        _RESULTS[qnum] = False
        print(f'Q{qnum}: ❌ Unanswered — replace ... with your answer.')
        return
    correct = predicted == actual
    _RESULTS[qnum] = correct
    if correct:
        print(f'Q{qnum}: ✅ Correct.')
    else:
        print(f'Q{qnum}: ❌ Incorrect.')
        print(f'  You answered: {predicted!r}')
        print(f'  Correct:      {actual!r}')
    if explain:
        for line in explain.split('\n'):
            print(f'   {line}')

def check_state(qnum: int, predicted, prog: str, state: dict | None = None,
                explain: str = '', max_steps: int = 10_000) -> None:
    """Run the program and compare final state."""
    if predicted is Ellipsis:
        _RESULTS[qnum] = False
        print(f'Q{qnum}: ❌ Unanswered — replace ... with a dict.')
        return
    actual = run(prog, state or {}, max_steps=max_steps)
    pred_canon = {k: v for k, v in predicted.items() if v != 0}
    check(qnum, pred_canon, actual, explain)

print('quiz harness ready')

## Section A — Syntax & Grammar

### Q1 — Valid arithmetic expression

Which of these is a valid `<aexp>` per the BNF grammar of While?

- **A.** `tt`
- **B.** `x + 1`
- **C.** `x = 0`
- **D.** `if 1 then 2 else 3`

In [ ]:
q1 = ...   # 'A', 'B', 'C', or 'D'
check(1, q1, 'B', explain=(
    'tt is a boolean expression, not an aexp.\n'
    'x = 0 is a boolean expression (the equality operator).\n'
    'if-then-else is a statement, not an expression.\n'
    'Only x + 1 matches <aexp> ::= <aexp> + <aexp>.'
))

### Q2 — NOT a valid boolean expression

Which of these is **NOT** a valid `<bexp>`?

- **A.** `x = 1`
- **B.** `tt`
- **C.** `x + 1`
- **D.** `!(x <= y)`

In [ ]:
q2 = ...   # 'A', 'B', 'C', or 'D'
check(2, q2, 'C', explain='x + 1 is an arithmetic expression, not a boolean. The bexp grammar has no "+" rule.')

### Q3 — Encode disjunction `a ∨ b`

Which is the correct encoding of `a ∨ b` in our ASCII syntax (where `a` and `b` are arbitrary boolean expressions)?

- **A.** `a | b`
- **B.** `a + b`
- **C.** `!((!a) & (!b))`
- **D.** `!(a & b)`

In [ ]:
q3 = ...
check(3, q3, 'C', explain=(
    "De Morgan: a ∨ b ≡ ¬(¬a ∧ ¬b).\n"
    "There's no | or + operator on booleans in While.\n"
    "!(a & b) is ¬(a ∧ b) — that's NAND, not OR."
))

### Q4 — Why is the BNF grammar ambiguous?

- **A.** Because it has too many statement types
- **B.** Because rules like `<aexp> ::= <aexp> + <aexp>` allow multiple parse trees for `1 + 2 + 3`
- **C.** Because it allows infinite recursion
- **D.** Because it doesn't define if-statements

In [ ]:
q4 = ...
check(4, q4, 'B', explain=(
    'The same string can be derived two ways: (1+2)+3 or 1+(2+3).\n'
    'Same kind of issue with chained ; (statements) and chained ∧ (booleans).'
))

## Section B — States and updates

### Q5 — State after multi-update

Compute the resulting state:

$$\{x \mapsto 1, y \mapsto 2\}[x \mapsto 0, z \mapsto 5]$$

Fill in the canonical form (omit zero-valued variables).

In [ ]:
# Type the resulting state as a Python dict — only non-zero entries.
q5 = ...   # e.g. {'a': 1, 'b': 2}

# The right answer:
actual5 = {'y': 2, 'z': 5}
check(5, q5, actual5, explain=(
    "x was 1, updated to 0 (so it drops out of the canonical form).\n"
    "y was 2 (untouched, kept).\n"
    "z gets 5 (new entry)."
))

## Section C — Evaluators A and B

### Q6 — Evaluate an arithmetic expression

What is `A((x + 2) * 3, {x: 4})`?

In [ ]:
q6 = ...   # an integer
check(6, q6, 18, explain='(4 + 2) * 3 = 6 * 3 = 18')

### Q7 — Evaluate a boolean expression

What is `B((x = 0) & !(y <= 3), {x: 0, y: 5})`?

In [ ]:
q7 = ...   # True or False
check(7, q7, True, explain=(
    'x = 0 with x = 0 is True.\n'
    'y <= 3 with y = 5 is False, so !(y <= 3) is True.\n'
    'True ∧ True is True.'
))

## Section D — Small-step rules

### Q8 — Assignment rule

Given `⟨x := 7, σ⟩`, the next configuration is:

- **A.** `⟨skip, σ[x ↦ 7]⟩`
- **B.** `⟨skip, σ⟩`
- **C.** `⟨x := 7, σ[x ↦ 7]⟩`
- **D.** `⟨skip, σ[x ↦ 0]⟩`

In [ ]:
q8 = ...
check(8, q8, 'A', explain=(
    'The := rule: ⟨x := a, σ⟩ ⇒ ⟨skip, σ[x ↦ A a σ]⟩.\n'
    'For a = 7 (a numeral), A(7, σ) = 7.'
))

### Q9 — `skip; T` rule

Given `⟨skip; T, σ⟩`, the next configuration is:

- **A.** `⟨skip, σ⟩`
- **B.** `⟨T, σ⟩`
- **C.** `⟨skip; T, σ⟩`
- **D.** `⟨T; skip, σ⟩`

In [ ]:
q9 = ...
check(9, q9, 'B', explain=(
    'The skip-; rule: ⟨skip; T, σ⟩ ⇒ ⟨T, σ⟩.\n'
    'Drop the skip from the left, state unchanged.'
))

### Q10 — `if` with false condition

Given `⟨if b then S else (S'), σ⟩` with `B(b, σ) = ff`, the next configuration is:

- **A.** `⟨S, σ⟩`
- **B.** `⟨S', σ⟩`
- **C.** `⟨skip, σ⟩`
- **D.** `⟨S; S', σ⟩`

In [ ]:
q10 = ...
check(10, q10, 'B', explain=(
    'The if-ff rule: when the condition is false, take the else-branch.\n'
    'State stays σ — checking a condition does not modify state.'
))

### Q11 — `while` with true condition

Given `⟨while b do (S), σ⟩` with `B(b, σ) = tt`, the next configuration is:

- **A.** `⟨skip, σ⟩`
- **B.** `⟨S, σ⟩`
- **C.** `⟨S; while b do (S), σ⟩`
- **D.** `⟨while b do (S), σ⟩`

In [ ]:
q11 = ...
check(11, q11, 'C', explain=(
    'The while-tt rule unfolds the loop: run body, then run the loop again.\n'
    'This is the rule that makes while-loops able to express unbounded computation —\n'
    'the program text grows during a loop iteration before it shrinks.'
))

## Section E — Tracing programs

### Q12 — Count transitions of a tiny program

How many small-step transitions does the program `x := 1; y := 2` take starting from `σ = {}`? Count every `⇒` arrow until you reach `⟨skip, σ'⟩`.

In [ ]:
q12 = ...   # an integer
check(12, q12, 3, explain=(
    'Step 1: ⟨x:=1; y:=2, σ⟩ ⇒ ⟨skip; y:=2, σ[x↦1]⟩  via ; rule (assignment fires inside)\n'
    'Step 2: ⇒ ⟨y:=2, σ[x↦1]⟩  via skip-; rule (drops the skip)\n'
    'Step 3: ⇒ ⟨skip, σ[x↦1, y↦2]⟩  via := rule (final assignment)\n'
    'Three transitions total. (Reaching skip ends the trace — skip itself is not a transition.)'
))

### Q13 — Final state of a small loop

What's the final state of

```
x := 0;
while x <= 2 do (x := x + 1)
```

starting from `σ = {}`? Give the canonical form (no zero-valued variables).

In [ ]:
q13 = ...   # a dict
check_state(13, q13,
    'x := 0; while x <= 2 do (x := x + 1)',
    explain=(
        'x goes 0 → 1 → 2 → 3.\n'
        'Iteration check at x = 3: 3 <= 2 is false, loop exits.\n'
        'Final: {x: 3}.'
    ))

### Q14 — Termination check

Does this program terminate from `σ = {}`?

```
x := 0;
while !(x = 5) do (x := x - 1)
```

In [ ]:
q14 = ...   # True (terminates) or False (doesn't)
check(14, q14, False, explain=(
    'x starts at 0 and decreases: 0, -1, -2, -3, ...\n'
    'It never equals 5 (we have arbitrary-precision integers, no underflow).\n'
    'The loop runs forever.'
))

## Section F — Reasoning about the semantics

### Q15 — State preservation (Exercise 17)

If `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` and `S` does **not** mention the variable `x`, then:

- **A.** `σ'(x) = σ(x)` — x is preserved
- **B.** `σ'(x) = 0`
- **C.** `σ'(x)` is undefined
- **D.** `σ'(x)` can be anything

In [ ]:
q15 = ...
check(15, q15, 'A', explain=(
    'Exercise 17. Proof by case analysis on which rule fires:\n'
    '  Assignment y := a only changes y (and y ≠ x since S has no x).\n'
    '  if/while rules leave the state unchanged.\n'
    '  skip-; rule leaves the state unchanged.\n'
    'In every case, σ\'(x) = σ(x).'
))

### Q16 — Associativity of `;` (Proposition 3)

What does Proposition 3 establish?

- **A.** Every While program terminates
- **B.** `⟨(S; T); U, σ⟩ ⇒* ⟨skip, τ⟩` iff `⟨S; (T; U), σ⟩ ⇒* ⟨skip, τ⟩`
- **C.** `;` is commutative — the order of statements doesn't matter
- **D.** `skip` is the identity element for `;`

In [ ]:
q16 = ...
check(16, q16, 'B', explain=(
    'Sequential composition is associative — how you parenthesize chained ; does not affect\n'
    'termination or the final state. This is what makes the BNF ambiguity for chained ;\n'
    'harmless. Note: ; is NOT commutative — order absolutely matters!'
))

### Q17 — Determinism (Exercise 18)

If `⟨S, σ⟩ ⇒ ⟨S', ρ⟩` and `⟨S, σ⟩ ⇒ ⟨S', τ⟩` then `ρ = τ`. This implies:

- **A.** While is non-deterministic — multiple next states are possible
- **B.** While is **deterministic** — each (S, σ) has at most one next configuration
- **C.** `skip` is the unique terminal program
- **D.** Variables can change in unpredictable ways

In [ ]:
q17 = ...
check(17, q17, 'B', explain=(
    'For any (S, σ) with S ≠ skip, exactly one rule applies, and it produces a uniquely\n'
    'determined (S\', σ\'). So the small-step relation is a partial function — non-deterministic\n'
    'languages would relax this; While does not.'
))

### Q18 — When does a program terminate?

A While program terminates when:

- **A.** The configuration reaches `⟨skip, σ⟩` for some σ
- **B.** All variables become 0
- **C.** After exactly n steps for some n
- **D.** The boolean condition of the outermost loop becomes false

In [ ]:
q18 = ...
check(18, q18, 'A', explain=(
    '⟨skip, σ⟩ is the terminal configuration — no rule applies, so no further transitions.\n'
    'B is wrong (variables can hold any int forever).\n'
    'C is wrong (could be any number of steps; some programs run forever).\n'
    'D is too narrow (a program can terminate without any while loops at all).'
))

### Q19 — Mixed program with if + assignment

What's the final state of

```
if x = 0 then
    y := 1
else (
    y := 2
);
z := y + 10
```

starting from `σ = {x: 5}`? Canonical form.

In [ ]:
q19 = ...   # a dict
check_state(19, q19,
    'if x = 0 then (y := 1) else (y := 2); z := y + 10',
    state={'x': 5},
    explain=(
        'x = 5, so x = 0 is false → if-ff fires → y := 2.\n'
        'Then z := y + 10 = 2 + 10 = 12.\n'
        'Final: {x: 5, y: 2, z: 12}.'
    ))

### Q20 — Iteration count

How many times does the **body** of this loop execute, starting from `σ = {x: 0}`?

```
while x <= 4 do (x := x + 2)
```

In [ ]:
q20 = ...   # an integer
check(20, q20, 3, explain=(
    'Iteration 1: x = 0, 0 ≤ 4 true → run body, x = 2.\n'
    'Iteration 2: x = 2, 2 ≤ 4 true → run body, x = 4.\n'
    'Iteration 3: x = 4, 4 ≤ 4 true → run body, x = 6.\n'
    'Check: x = 6, 6 ≤ 4 false → exit. Body ran 3 times.'
))

## Final score

Run this cell once you've answered all 20 questions.

In [ ]:
total = len(_RESULTS)
correct = sum(_RESULTS.values())
missing = [n for n in range(1, 21) if n not in _RESULTS]

print(f'Score: {correct}/{total}')
if missing:
    print(f'  (you have not answered: {missing})')

if total == 0:
    print('No answers recorded — go run the question cells.')
elif correct == 20:
    print('🎉 Perfect — every concept covered. You are ready.')
elif correct >= 18:
    print('Strong grasp. Spot-check the questions you missed and you are exam-ready.')
elif correct >= 15:
    print('Solid. Re-read the section(s) the missed questions came from.')
elif correct >= 12:
    print('Mid-pack. Worth one more pass through N3 (semantics) before the exam.')
else:
    print('Worth a full re-read. Start again at N1, slow down on predict-cells.')

print()
print('Question-by-question:')
section_map = {
    'A — Syntax & Grammar':            range(1, 5),
    'B — States and updates':          range(5, 6),
    'C — Evaluators A and B':          range(6, 8),
    'D — Small-step rules':            range(8, 12),
    'E — Tracing programs':            list(range(12, 15)) + [19, 20],
    'F — Reasoning about the semantics': range(15, 19),
}
for section, qnums in section_map.items():
    qnums = list(qnums)
    sec_correct = sum(_RESULTS.get(q, False) for q in qnums)
    sec_total = len(qnums)
    marks = ' '.join('✅' if _RESULTS.get(q) else ('❌' if q in _RESULTS else '·') for q in qnums)
    print(f'  {section}: {sec_correct}/{sec_total}   {marks}')

## Where to go from here

If you missed questions in:

- **Section A (syntax)** → re-read N2 §1–4. The BNF rules are the foundation.
- **Section B (states)** → re-read N3 §1–2 plus Exercise 11.
- **Section C (A and B)** → re-read N3 §3 — the truth tables are short, and they recur in every question that involves an if or while.
- **Section D (rules)** → this is the heart of the material. Re-read N3 §4 carefully. Try producing the formal trace for Example 1 from scratch on paper.
- **Section E (tracing)** → run more programs through `trace(prog, σ, view='formal')`. Predict each step before the interpreter shows you.
- **Section F (reasoning)** → re-read N4 §1–3. The proofs are by case-analysis on rules; once that pattern clicks, the answers feel mechanical.